# 06 — Item-based product recommendations

Select a product by **parent_asin** and return 10 similar products, prioritizing category-compatible items and filling shortages from related categories. This is content-based item-to-item recommendation using the existing E5 passage vectors, CLIP image vectors, and FAISS indexes from notebooks 03–04. There are no encoder downloads, embedding generation, index builds, or dependencies on running earlier notebooks in the same session.

Run in **Google Colab**, CPU is sufficient. All inputs come from Google Drive's `ElectroLens/search_assets/`. Only configuration, evaluation, and human judgment files are saved to `ElectroLens/recommendation_assets/`, with stable filenames and no copied assets or versioned folders.

Notebooks 01–02 define `parent_asin`, `product_group`, `subcategory`, `cat_path`, and product metadata. Notebook 03 preserves catalog row order and saves normalized vectors; notebook 04 saves E5 and CLIP indexes and the CLIP row mapping. Notebook 05's search labels judge **text query/product** pairs, so they cannot serve as item-pair relevance labels. Reviews contain review IDs and product IDs but no user IDs; this notebook does not infer co-purchase behavior or relevance from review counts/ratings.


## 1 · Colab setup


In [1]:
%pip -q install "faiss-cpu>=1.8" numpy pandas pyarrow


In [2]:
import json, re, math, hashlib, html, unicodedata, platform
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations
from importlib.metadata import version

import numpy as np
import pandas as pd
import faiss
from IPython.display import HTML, display
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/ElectroLens')
SEARCH = ROOT / 'search_assets'
SAVE = ROOT / 'recommendation_assets'
SEED, TOP_K, SAMPLES_PER_GROUP = 42, 10, 3

# Fixed design settings, not learned or tuned on the evaluation samples.
RANKING = {'candidate_k': 200, 'text_weight': 0.65, 'image_weight': 0.35,
           'category_weight': 0.10}
METHODS = ('text', 'image', 'hybrid')
print('Read:', SEARCH, '| Save reports only to:', SAVE)


Mounted at /content/drive
Read: /content/drive/MyDrive/ElectroLens/search_assets | Save reports only to: /content/drive/MyDrive/ElectroLens/recommendation_assets


## 2 · Validate assets and row mappings

Use `products_search.parquet` as the canonical ordered catalog; do not replace it with the local or Drive `products_final.parquet` and assume row order matches. Expected model IDs/dimensions are from notebook 03. E5 FAISS positions equal catalog rows. CLIP FAISS positions map through `clip_image_row_ids.npy`; unavailable image rows contain zeros and are never queried as images.

Validation checks all vectors in small blocks for finite values and normalization, plus sampled index reconstruction against the saved arrays. No vectors are added to either index. A mismatch stops the notebook; it never creates replacement artifacts. BM25 is not needed for selected-item similarity.


In [3]:
def require(condition, message):
    if not condition:
        raise ValueError(message)

def text_value(value):
    if value is None or pd.isna(value):
        return ''
    return re.sub(r'\s+', ' ', str(value)).strip()

ASSET_FILES = ['manifest.json', 'products_search.parquet', 'e5_embeddings.npy',
               'clip_image_embeddings.npy', 'clip_image_valid.npy',
               'e5_text.faiss', 'clip_image.faiss', 'clip_image_row_ids.npy']
missing = [name for name in ASSET_FILES if not (SEARCH / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing existing Drive assets: {missing}. Check the Drive mount/path.')

products = pd.read_parquet(SEARCH / 'products_search.parquet').reset_index(drop=True)
needed = {'parent_asin', 'title', 'product_group', 'subcategory', 'cat_path', 'image_url'}
require(needed <= set(products), f'Catalog requires columns: {sorted(needed)}')
require(len(products) > 1 and products.parent_asin.notna().all()
        and products.parent_asin.is_unique, 'Catalog must have unique non-null product IDs')
products['parent_asin'] = products.parent_asin.astype(str)
asins = products.parent_asin.to_numpy()
require(products.parent_asin.str.fullmatch(r'[A-Z0-9]{10}').all(), 'Unexpected ASIN format')
for column in ('title', 'product_group', 'cat_path'):
    require(products[column].map(text_value).ne('').all(), f'Missing {column} values')
require(products.cat_path.map(lambda p: len(str(p).split(' > ')) >= 3).all(),
        'Category paths differ from notebook 01 schema')

manifest = json.loads((SEARCH / 'manifest.json').read_text(encoding='utf-8'))
catalog_hash = hashlib.sha256('\n'.join(asins).encode()).hexdigest()[:16]
require(catalog_hash == manifest['catalog_hash'] and len(products) == manifest['n_products'],
        'Catalog product ID/order fingerprint mismatch')
require(manifest['e5_model'] == 'intfloat/multilingual-e5-small'
        and manifest['clip_model'] == 'openai/clip-vit-base-patch32', 'Unexpected saved model IDs')

e5_vectors = np.load(SEARCH / 'e5_embeddings.npy', mmap_mode='r', allow_pickle=False)
image_vectors = np.load(SEARCH / 'clip_image_embeddings.npy', mmap_mode='r', allow_pickle=False)
image_valid = np.load(SEARCH / 'clip_image_valid.npy', allow_pickle=False)
image_rows = np.load(SEARCH / 'clip_image_row_ids.npy', allow_pickle=False)
e5_index = faiss.read_index(str(SEARCH / 'e5_text.faiss'))
image_index = faiss.read_index(str(SEARCH / 'clip_image.faiss'))

require(e5_vectors.shape == (len(products), 384) and e5_vectors.dtype == np.float32,
        'Unexpected E5 shape/dtype')
require(image_vectors.shape == (len(products), 512) and image_vectors.dtype == np.float32,
        'Unexpected CLIP shape/dtype')
require(image_valid.shape == (len(products),) and image_valid.dtype == np.bool_,
        'Invalid image availability mask')
require(image_rows.ndim == 1 and np.issubdtype(image_rows.dtype, np.integer)
        and np.array_equal(image_rows, np.flatnonzero(image_valid)), 'CLIP row mapping differs')
require(e5_index.ntotal == len(products) and e5_index.d == 384
        and image_index.ntotal == len(image_rows) and image_index.d == 512, 'FAISS shape/count mismatch')
require(e5_index.metric_type == faiss.METRIC_INNER_PRODUCT
        and image_index.metric_type == faiss.METRIC_INNER_PRODUCT, 'Expected inner-product FAISS indexes')

for start in range(0, len(products), 1024):
    end = min(start + 1024, len(products))
    text_block = np.asarray(e5_vectors[start:end])
    image_block = np.asarray(image_vectors[start:end])
    valid = image_valid[start:end]
    require(np.isfinite(text_block).all() and np.isfinite(image_block).all(), 'Non-finite vectors')
    require(np.allclose(np.linalg.norm(text_block, axis=1), 1.0, atol=1e-3), 'E5 vectors not normalized')
    if valid.any():
        require(np.allclose(np.linalg.norm(image_block[valid], axis=1), 1.0, atol=1e-3),
                'Valid CLIP vectors not normalized')
    require(np.all(image_block[~valid] == 0), 'Invalid image rows must contain zero vectors')

mapping_checks = []
for name, index, vectors, rows in [('e5', e5_index, e5_vectors, np.arange(len(products))),
                                    ('clip', image_index, image_vectors, image_rows)]:
    if not index.ntotal:
        continue
    for position in np.linspace(0, index.ntotal - 1, min(32, index.ntotal), dtype=int):
        row = int(rows[position])
        require(np.allclose(index.reconstruct(int(position)), vectors[row], atol=1e-6),
                f'{name} index/embedding row mismatch')
        mapping_checks.append({'index': name, 'index_position': int(position),
                               'catalog_row': row, 'parent_asin': str(asins[row])})

display(products.groupby('product_group').size().rename('products').to_frame())
print('Catalog:', len(products), '| valid images:', int(image_valid.sum()), '| fingerprint:', catalog_hash)


,products
product_group,
accessories,1105
camera,1216
desktop,1092
headphones,1195
keyboard,1072
laptop,1060
monitor,1065
mouse,1053
router,976


Catalog: 13328 | valid images: 13328 | fingerprint: 4f1e28ecb05ad3eb


## 3 · Similarity, category, and duplicate policy

- Retrieve neighbors from the saved FAISS indexes using the selected item's **saved product vectors**, with no E5 query prefix or re-encoding.
- First-priority main devices share `product_group`; exact leaf matches score 1, a specific shared parent 0.85, other same-group products 0.60.
- For first-priority accessory matches, the broad `accessories` group alone is insufficient. Require the same full category path, or a shared specific parent and informative leaf terms. Ancestor/generic words are removed before sibling matching. Category compatibility is a taxonomy rule, not electrical, connector, or model-fit verification.
- If global neighbors underfill a compatible candidate pool, use exact dot products on existing vectors in the compatible subset. This recovers compatible candidates missed by global retrieval without rebuilding an index.
- Hybrid similarity is 65% E5 + 35% CLIP when both products have valid images. Otherwise use E5 alone. Each cosine is mapped to `(cosine + 1) / 2`; the within-tier score is 90% similarity + 10% category compatibility/relatedness. Weights are fixed heuristics, not calibrated probabilities. Renormalizing missing-image weights can change score distributions; compare availability strata separately.
- Exclude the seed, repeated ASINs, and exact normalized-title duplicates (including aliases of the seed). Keep distinct model/color titles; near-duplicate detection is limited. If compatible products provide fewer than 10 distinct titles, exhaust them first, then fill from related categories: shared specific ancestry/leaf terms, shared department, and finally the closest semantic neighbors elsewhere in Electronics. Rank each tier by similarity score and ASIN for ties; a higher fallback score never displaces a compatible product. Exact scoring of the required tiers avoids candidate-cap shortages. Each result exposes its tier and fallback source. A shorter list occurs only if the entire available catalog has fewer than 10 distinct alternatives. Image-only never silently falls back to text.

All three methods use the same compatibility, deduplication, and category-score rules. There is no diversity reranking or weight tuning hidden in the comparison.

Strict category compatibility and fallback relatedness are separate fields. Display order is `(category_tier, -score, parent_asin)`, so scores need not descend across tier boundaries. No fallback is a verified connector/model fit.


In [4]:
def normalized_text(value):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', text_value(value)).casefold()).strip()

def title_key(value):
    return ' '.join(re.findall(r'\w+', normalized_text(value), flags=re.UNICODE))

def category_terms(value):
    terms = set()
    for token in re.findall(r'\w+', normalized_text(value)):
        if token.endswith('ies') and len(token) > 4:
            token = token[:-3] + 'y'
        elif token.endswith('s') and len(token) > 4 and not token.endswith(('ss', 'us', 'is')):
            token = token[:-1]
        terms.add(token)
    return terms - {'and', 'the', 'for', 'with', 'accessory', 'supply', 'other', 'electronic'}

class ItemRecommender:
    """Read-only recommendation service; request methods do not mount Drive or write files."""

    def __init__(self, catalog, text_vectors, image_vectors, image_valid,
                 text_index, image_index, image_rows, config):
        self.catalog = catalog
        self.text_vectors, self.image_vectors = text_vectors, image_vectors
        self.image_valid = image_valid
        self.text_index, self.image_index, self.image_rows = text_index, image_index, image_rows
        self.config = dict(config)
        require(isinstance(self.config['candidate_k'], int) and self.config['candidate_k'] > 0,
                'candidate_k must be positive')
        tw, iw, cw = (self.config[k] for k in ('text_weight', 'image_weight', 'category_weight'))
        require(all(np.isfinite(v) for v in (tw, iw, cw))
                and tw > 0 and iw > 0 and np.isclose(tw + iw, 1) and 0 <= cw <= 1, 'Invalid weights')
        self.ids = catalog.parent_asin.astype(str).tolist()
        self.row_by_id = {asin: row for row, asin in enumerate(self.ids)}
        self.groups = [normalized_text(v) for v in catalog.product_group]
        self.paths = [tuple(normalized_text(v) for v in str(p).split(' > ') if text_value(v))
                      for p in catalog.cat_path]
        self.leaves = [normalized_text(s) or p[-1] for s, p in zip(catalog.subcategory, self.paths)]
        self.duplicate_keys = [title_key(t) or self.ids[i] for i, t in enumerate(catalog.title)]
        self.leaf_terms = [
            category_terms(leaf) - category_terms(' '.join(path[:-1]))
            for leaf, path in zip(self.leaves, self.paths)]
        self.group_rows = defaultdict(list)
        for row, group in enumerate(self.groups):
            self.group_rows[group].append(row)
        self.available_title_keys = {
            'all': frozenset(self.duplicate_keys),
            'image': frozenset(key for row, key in enumerate(self.duplicate_keys) if self.image_valid[row])}

    @staticmethod
    def validate_top_k(top_k):
        if isinstance(top_k, bool) or not isinstance(top_k, int) or not 1 <= top_k <= 100:
            raise ValueError('top_k must be an integer between 1 and 100')

    def compatibility(self, seed, candidate):
        if self.groups[seed] != self.groups[candidate]:
            return 0.0
        left, right = self.paths[seed], self.paths[candidate]
        same_parent = len(left[:-1]) >= 3 and left[:-1] == right[:-1]
        if self.groups[seed] == 'accessories':
            if left == right:
                return 1.0
            return 0.75 if same_parent and self.leaf_terms[seed] & self.leaf_terms[candidate] else 0.0
        if self.leaves[seed] == self.leaves[candidate]:
            return 1.0
        return 0.85 if same_parent else 0.60

    def category_relation(self, seed, candidate):
        compatibility = self.compatibility(seed, candidate)
        if compatibility > 0:
            return 0, compatibility
        left, right = self.paths[seed], self.paths[candidate]
        shared = 0
        for a, b in zip(left, right):
            if a != b:
                break
            shared += 1
        if shared >= 3 or (shared >= 2 and self.leaf_terms[seed] & self.leaf_terms[candidate]):
            return 1, 0.50
        if shared >= 2:
            return 2, 0.25
        # Last resort: semantically closest products elsewhere in Electronics.
        # This is explicitly marked as broad fallback, never as strict compatibility.
        return 3, 0.10 if shared >= 1 else 0.0

    def related_candidate_rows(self, seed, method):
        tiers = defaultdict(list)
        for row in range(len(self.catalog)):
            if row == seed or self.duplicate_keys[row] == self.duplicate_keys[seed]:
                continue
            if method == 'image' and not self.image_valid[row]:
                continue
            tier, _ = self.category_relation(seed, row)
            if tier > 0:
                tiers[tier].append(row)
        return tiers

    def eligible_rows(self, seed, method):
        return [
            row for row in self.group_rows[self.groups[seed]]
            if row != seed and self.duplicate_keys[row] != self.duplicate_keys[seed]
            and self.compatibility(seed, row) > 0
            and (method != 'image' or (self.image_valid[seed] and self.image_valid[row]))]

    def product(self, row):
        product = self.catalog.iloc[row]
        price = product.get('price')
        price = float(price) if price is not None and pd.notna(price) and np.isfinite(price) else None
        return {'catalog_row': int(row), 'parent_asin': self.ids[row],
                'title': text_value(product['title']), 'product_group': text_value(product['product_group']),
                'subcategory': text_value(product['subcategory']), 'cat_path': text_value(product['cat_path']),
                'price': price, 'image_url': text_value(product.get('image_url')),
                'image_embedding_available': bool(self.image_valid[row])}

    def candidate_rows(self, seed, method, top_k, eligible):
        neighbors = set()
        k = self.config['candidate_k'] + 1  # The selected item can occupy a neighbor position.
        if method in ('text', 'hybrid'):
            query = np.ascontiguousarray(self.text_vectors[seed:seed + 1], dtype='float32')
            _, ids = self.text_index.search(query, min(k, self.text_index.ntotal))
            neighbors.update(int(i) for i in ids[0] if i >= 0)
        if method in ('image', 'hybrid') and self.image_valid[seed] and self.image_index.ntotal:
            query = np.ascontiguousarray(self.image_vectors[seed:seed + 1], dtype='float32')
            _, ids = self.image_index.search(query, min(k, self.image_index.ntotal))
            neighbors.update(int(self.image_rows[int(i)]) for i in ids[0] if i >= 0)
        neighbors.intersection_update(eligible)
        possible = len({self.duplicate_keys[row] for row in eligible})
        wanted_pool = min(possible, max(50, 5 * top_k))
        unique_neighbors = len({self.duplicate_keys[row] for row in neighbors})
        backfill = unique_neighbors < wanted_pool
        if backfill:
            # Typically <=1,500 same-group rows from notebook 01. Score saved vectors only.
            neighbors.update(eligible)
        return sorted(neighbors), possible, backfill

    def score_candidates(self, seed, rows, method):
        text_cosines = np.clip(self.text_vectors[rows] @ self.text_vectors[seed], -1.0, 1.0)
        image_cosines = np.full(len(rows), np.nan, dtype='float32')
        available = self.image_valid[rows] & bool(self.image_valid[seed])
        if available.any():
            valid_rows = np.asarray(rows, dtype='int64')[available]
            image_cosines[available] = np.clip(
                self.image_vectors[valid_rows] @ self.image_vectors[seed], -1.0, 1.0)
        scored = []
        tw, iw = self.config['text_weight'], self.config['image_weight']
        cw = self.config['category_weight']
        for offset, row in enumerate(rows):
            text_cosine = float(text_cosines[offset])
            image_cosine = float(image_cosines[offset]) if available[offset] else None
            text_score = (text_cosine + 1.0) / 2.0
            if method == 'image':
                similarity, modalities = (image_cosine + 1.0) / 2.0, ['image']
            elif method == 'hybrid' and image_cosine is not None:
                similarity = tw * text_score + iw * (image_cosine + 1.0) / 2.0
                modalities = ['text', 'image']
            else:
                similarity, modalities = text_score, ['text']
            category = self.compatibility(seed, row)
            tier, relatedness = self.category_relation(seed, row)
            source = ('compatible', 'related_category', 'same_department', 'catalog_similarity')[tier]
            scored.append({'catalog_row': int(row), 'score': float((1 - cw) * similarity + cw * relatedness),
                           'similarity_score': float(similarity), 'text_similarity': text_cosine,
                           'image_similarity': image_cosine, 'category_compatibility': float(category),
                           'category_relatedness': float(relatedness), 'category_tier': int(tier),
                           'recommendation_source': source, 'is_category_fallback': bool(tier > 0),
                           'modalities_used': modalities})
        return sorted(scored, key=lambda item: (item['category_tier'], -item['score'],
                                               self.ids[item['catalog_row']]))

    def recommend(self, parent_asin, top_k=10, method='hybrid'):
        self.validate_top_k(top_k)
        if method not in ('text', 'image', 'hybrid'):
            raise ValueError('method must be text, image, or hybrid')
        if not isinstance(parent_asin, str) or not re.fullmatch(r'[A-Z0-9]{10}', parent_asin.strip().upper()):
            return {'status': 'invalid_product_id', 'selected_product': None, 'results': [],
                    'diagnostics': {'method': method, 'requested_k': top_k}}
        asin = parent_asin.strip().upper()
        seed = self.row_by_id.get(asin)
        if seed is None:
            return {'status': 'product_not_found', 'selected_product': None, 'results': [],
                    'diagnostics': {'method': method, 'requested_k': top_k}}
        selected = self.product(seed)
        diagnostics = {'method': method, 'requested_k': top_k, 'selected_image_available': bool(self.image_valid[seed]),
                       'image_fallback_to_text': bool(method == 'hybrid' and not self.image_valid[seed])}
        if method == 'image' and not self.image_valid[seed]:
            return {'status': 'image_unavailable', 'selected_product': selected, 'results': [],
                    'diagnostics': diagnostics}
        eligible = self.eligible_rows(seed, method)
        rows, possible, backfill = self.candidate_rows(seed, method, top_k, eligible)
        seen_ids, seen_titles, results = {asin}, {self.duplicate_keys[seed]}, []

        def append_unique(scored):
            for item in scored:
                if len(results) >= top_k:
                    break
                row = item['catalog_row']
                if self.ids[row] in seen_ids or self.duplicate_keys[row] in seen_titles:
                    continue
                seen_ids.add(self.ids[row])
                seen_titles.add(self.duplicate_keys[row])
                results.append({**self.product(row), **item, 'rank': len(results) + 1})

        append_unique(self.score_candidates(seed, rows, method) if rows else [])
        primary_count = len(results)
        related_candidates_scored = 0
        if len(results) < top_k:
            # Exhaust compatible titles before widening to related taxonomy branches.
            # Score every remaining row in each necessary tier using saved vectors;
            # no fixed global-neighbor cap can prevent filling after deduplication.
            for tier, tier_rows in sorted(self.related_candidate_rows(seed, method).items()):
                tier_rows = [row for row in tier_rows if self.duplicate_keys[row] not in seen_titles]
                if tier_rows:
                    related_candidates_scored += len(tier_rows)
                    append_unique(self.score_candidates(seed, tier_rows, method))
                if len(results) == top_k:
                    break
        universe = self.available_title_keys['image' if method == 'image' else 'all']
        available = len(universe - {self.duplicate_keys[seed]})
        diagnostics.update(candidate_count=len(rows) + related_candidates_scored,
                           eligible_distinct_products=possible, available_distinct_products=available,
                           category_backfill_used=bool(backfill), returned=len(results),
                           compatible_returned=primary_count, related_category_fill_count=len(results) - primary_count,
                           related_categories_used=sorted({r['recommendation_source'] for r in results
                                                          if r['is_category_fallback']}),
                           result_text_fallback_count=sum(
                               r['modalities_used'] == ['text'] for r in results) if method == 'hybrid' else 0)
        status = 'ok' if len(results) == top_k else 'insufficient_distinct_products'
        return {'status': status, 'selected_product': selected, 'results': results, 'diagnostics': diagnostics}

recommender = ItemRecommender(products, e5_vectors, image_vectors, image_valid,
                              e5_index, image_index, image_rows, RANKING)

def recommend_products(parent_asin, top_k=10):
    """FastAPI-ready entry point: selected product, recommendations, status, and diagnostics."""
    return recommender.recommend(parent_asin, top_k=top_k, method='hybrid')


## 4 · Reproducible real-catalog samples and modality comparison

Choose three catalog items per existing product group using a fixed RNG and ASIN-sorted sampling. Add one missing-image item per group where available. Samples are selected before examining recommendation quality. They are group-balanced diagnostics, not an estimate of user traffic or a full-catalog benchmark. Persist the selected IDs/rows in the evaluation report.

Run all three methods for the **same seeds**. The image-available subset provides a paired modality comparison; the all-seed view also reveals missing-image behavior. Only metadata and saved vectors are used. These calculations run in Colab, not against nonexistent local indexes.


In [5]:
rng = np.random.default_rng(SEED)
sample_rows = set()
for group in sorted(recommender.group_rows):
    rows = sorted(recommender.group_rows[group], key=lambda r: recommender.ids[r])
    sample_rows.update(int(r) for r in rng.choice(rows, size=min(SAMPLES_PER_GROUP, len(rows)), replace=False))
    missing_image_rows = [r for r in rows if not image_valid[r]]
    if missing_image_rows:
        sample_rows.add(int(rng.choice(missing_image_rows)))
sample_rows = sorted(sample_rows, key=lambda r: (recommender.groups[r], recommender.ids[r]))
sample_df = pd.DataFrame([recommender.product(row) for row in sample_rows])
display(sample_df[['catalog_row', 'parent_asin', 'title', 'product_group', 'subcategory',
                   'image_embedding_available']])
print('Sample seeds:', len(sample_rows), '| with images:', int(sample_df.image_embedding_available.sum()))

runs = {
    (recommender.ids[row], method): recommender.recommend(recommender.ids[row], TOP_K, method=method)
    for row in sample_rows for method in METHODS}
display(pd.DataFrame([
    {'parent_asin': asin, 'method': method, 'status': response['status'],
     'returned': len(response['results']), 'image_available': response['selected_product']['image_embedding_available'],
     'category_backfill': response['diagnostics'].get('category_backfill_used', False)}
    for (asin, method), response in runs.items()]))


,catalog_row,parent_asin,title,product_group,subcategory,image_embedding_available
0,13036,B001KNP3MQ,DGK Color Tools Optek Premium Reference White ...,accessories,Color Calibration Charts,True
1,12901,B07QT88378,SVBONY Telescope Eyepiece 34mm 2 inches Ultra ...,accessories,Eyepieces,True
2,12248,B08FGD53GP,"LuBanSir Cinch Straps, Assorted 9 Pack（6“/9”/1...",accessories,Cable Straps,True
3,8501,B000MANEDI,JVC Everio GZMG555 5MP 30GB Hard Disk Drive Ca...,camera,Camcorders,True
4,8343,B088WTLBQR,Canon EOS Rebel T7 DSLR Camera Bundle with Can...,camera,DSLR Cameras,True
5,7824,B0B1M91G7W,"DEPSTECH Webcam with Microphone, 4K Webcam Son...",camera,Webcams,True
6,3171,B089YX5QJ6,"Dell Optiplex 7020 Desktop Computer PC, Intel ...",desktop,Towers,True
7,3060,B0BCQD7KWP,HP 2022 TP01 Pavilion Business Desktop Intel 6...,desktop,Minis,True
8,2426,B0CBD98MY3,Dell Optiplex 7040 Business SFF Computer Small...,desktop,Towers,True
9,617,B01JP436TS,Sennheiser HD 598 Cs Closed Back Headphone,headphones,Over-Ear Headphones,True


Sample seeds: 39 | with images: 39


,parent_asin,method,status,returned,image_available,category_backfill
0,B001KNP3MQ,text,ok,10,True,True
1,B001KNP3MQ,image,ok,10,True,True
2,B001KNP3MQ,hybrid,ok,10,True,True
3,B07QT88378,text,ok,10,True,False
4,B07QT88378,image,ok,10,True,True
...,...,...,...,...,...,...
112,B0B3S7JVND,image,ok,10,True,False
113,B0B3S7JVND,hybrid,ok,10,True,False
114,B0BKWSLTHK,text,ok,10,True,False
115,B0BKWSLTHK,image,ok,10,True,False


## 5 · Display titles, categories, scores, and images


In [6]:
def image_tag(url, title):
    url = text_value(url)
    if not url.lower().startswith(('https://', 'http://')):
        return '<span>Image unavailable</span>'
    return (f'<img src="{html.escape(url, quote=True)}" alt="{html.escape(title, quote=True)}" '
            'loading="lazy" referrerpolicy="no-referrer" '
            'style="width:120px;height:100px;object-fit:contain">')

def show_recommendations(response):
    selected = response['selected_product']
    if selected:
        display(HTML('<div style="display:flex;gap:16px;align-items:center">'
                     + image_tag(selected['image_url'], selected['title'])
                     + '<div><b>Selected: ' + html.escape(selected['title']) + '</b><br>'
                     + html.escape(selected['cat_path']) + '<br>'
                     + html.escape(selected['parent_asin']) + '</div></div>'))
    print('Status:', response['status'], '|', response['diagnostics'])
    rows = []
    for item in response['results']:
        rows.append({
            'Rank': item['rank'], 'Image': image_tag(item['image_url'], item['title']),
            'ASIN': html.escape(item['parent_asin']), 'Title': html.escape(item['title']),
            'Group': html.escape(item['product_group']), 'Category': html.escape(item['subcategory']),
            'Score': round(item['score'], 4), 'E5 cosine': round(item['text_similarity'], 4),
            'CLIP cosine': round(item['image_similarity'], 4) if item['image_similarity'] is not None else 'Unavailable',
            'Category compatibility': item['category_compatibility'],
            'Source': item['recommendation_source'],
            'Used': ', '.join(item['modalities_used'])})
    if rows:
        display(HTML(pd.DataFrame(rows).to_html(index=False, escape=False)))

# Default: one reproducible sample seed, three methods (30 thumbnails at most).
# Change DEMO_ASIN to any valid catalog ID or call recommend_products directly.
DEMO_ASIN = recommender.ids[sample_rows[0]]
for method in METHODS:
    print('\nMETHOD:', method)
    show_recommendations(runs[(DEMO_ASIN, method)])

# Inspect real features/description before judging relevance when needed.
def inspect_product(parent_asin):
    row = recommender.row_by_id.get(str(parent_asin).strip().upper())
    if row is None:
        print('Product not found')
        return
    fields = [c for c in ('parent_asin', 'title', 'cat_path', 'price', 'features', 'description')
              if c in products.columns]
    display(products.iloc[[row]][fields].T)



METHOD: text


Status: ok | {'method': 'text', 'requested_k': 10, 'selected_image_available': True, 'image_fallback_to_text': False, 'candidate_count': 49, 'eligible_distinct_products': 8, 'available_distinct_products': 13211, 'category_backfill_used': True, 'returned': 10, 'compatible_returned': 8, 'related_category_fill_count': 2, 'related_categories_used': ['related_category'], 'result_text_fallback_count': 0}


Rank,Image,ASIN,Title,Group,Category,Score,E5 cosine,CLIP cosine,Category compatibility,Source,Used
1,,B00DMA06AY,"DGK Color Tools DKC-Pro 5"" x 7"" Set of 2 White Balance and Color Calibration Charts with 12% and 18% Gray - Pro Quality - Includes Frame Stand and User Guide",accessories,Color Calibration Charts,0.9653,0.9228,0.6784,1.0,compatible,text
2,,B00EGBIZ2O,"DGK Color Tools Digital Kolor Pro 16:9 Large Color Calibration and Video Chip Chart, 2-Pack",accessories,Color Calibration Charts,0.9423,0.8717,0.6774,1.0,compatible,text
3,,B07VCTWR5Q,"Pixel Perfect Camera Color Correction Card – (Set of 2) - 4"" x 6"" for Photo and Video - Reference Tool Grey Card Target White Balance Exposure Temperature Color Calibration Chart",accessories,Color Calibration Charts,0.9373,0.8606,0.5647,1.0,compatible,text
4,,B0C5N8B857,Calibrite ColorChecker Digital SG (CCDSG),accessories,Color Calibration Charts,0.9344,0.8543,0.6242,1.0,compatible,text
5,,B07QR6J48J,X-Rite ColorChecker Passport Photo (MSCCPP),accessories,Color Calibration Charts,0.9275,0.8389,0.7630,1.0,compatible,text
6,,B08RDK4TPR,Datacolor SpyderCHECKR 24 - Color calibrate your camera for consistent image color across multiple camera systems/lighting conditions. Target color chart has 24 target colors + grey card.,accessories,Color Calibration Charts,0.9272,0.8383,0.6744,1.0,compatible,text
7,,B0C2ZRC4PV,Datacolor Spyder X Capture Pro: Photo Tool Kit for Precision Color Control from Capture Through Editing,accessories,Color Calibration Charts,0.9271,0.8380,0.6060,1.0,compatible,text
8,,B07QNYHD9H,X-Rite i1Display Pro - Display Calibration (EODIS3),accessories,Color Calibration Charts,0.9198,0.8219,0.6306,1.0,compatible,text
9,,B01B4GT7VG,"Kiorafoto 3""x3"" Easy Carry Professional Camera White Balance Card Disc Grey Card Film Color Correction Filter Color Checker Temperature Calibration Tool with Neck Strap String for Lens up to 82mm",accessories,Color Correction & Compensation Filters,0.8930,0.8734,0.6919,0.0,related_category,text
10,,B00GSC97HG,"Movo Photo Color, White Balance Card Set for Digital Photography (Index Card Sized, 5 X 4)",accessories,Grey Cards,0.8921,0.8714,0.7582,0.0,related_category,text



METHOD: image


Status: ok | {'method': 'image', 'requested_k': 10, 'selected_image_available': True, 'image_fallback_to_text': False, 'candidate_count': 49, 'eligible_distinct_products': 8, 'available_distinct_products': 13211, 'category_backfill_used': True, 'returned': 10, 'compatible_returned': 8, 'related_category_fill_count': 2, 'related_categories_used': ['related_category'], 'result_text_fallback_count': 0}


Rank,Image,ASIN,Title,Group,Category,Score,E5 cosine,CLIP cosine,Category compatibility,Source,Used
1,,B07QR6J48J,X-Rite ColorChecker Passport Photo (MSCCPP),accessories,Color Calibration Charts,0.8934,0.8389,0.7630,1.0,compatible,image
2,,B00DMA06AY,"DGK Color Tools DKC-Pro 5"" x 7"" Set of 2 White Balance and Color Calibration Charts with 12% and 18% Gray - Pro Quality - Includes Frame Stand and User Guide",accessories,Color Calibration Charts,0.8553,0.9228,0.6784,1.0,compatible,image
3,,B00EGBIZ2O,"DGK Color Tools Digital Kolor Pro 16:9 Large Color Calibration and Video Chip Chart, 2-Pack",accessories,Color Calibration Charts,0.8548,0.8717,0.6774,1.0,compatible,image
4,,B08RDK4TPR,Datacolor SpyderCHECKR 24 - Color calibrate your camera for consistent image color across multiple camera systems/lighting conditions. Target color chart has 24 target colors + grey card.,accessories,Color Calibration Charts,0.8535,0.8383,0.6744,1.0,compatible,image
5,,B07QNYHD9H,X-Rite i1Display Pro - Display Calibration (EODIS3),accessories,Color Calibration Charts,0.8338,0.8219,0.6306,1.0,compatible,image
6,,B0C5N8B857,Calibrite ColorChecker Digital SG (CCDSG),accessories,Color Calibration Charts,0.8309,0.8543,0.6242,1.0,compatible,image
7,,B0C2ZRC4PV,Datacolor Spyder X Capture Pro: Photo Tool Kit for Precision Color Control from Capture Through Editing,accessories,Color Calibration Charts,0.8227,0.8380,0.6060,1.0,compatible,image
8,,B07VCTWR5Q,"Pixel Perfect Camera Color Correction Card – (Set of 2) - 4"" x 6"" for Photo and Video - Reference Tool Grey Card Target White Balance Exposure Temperature Color Calibration Chart",accessories,Color Calibration Charts,0.8041,0.8606,0.5647,1.0,compatible,image
9,,B073XPTZJW,"Anwenk Grey Card White Balance Card 18% Exposure Photography Card Custom Calibration Camera Checker Video, DSLR and Film",accessories,Grey Cards,0.8698,0.8642,0.8218,0.0,related_category,image
10,,B00GSC97HG,"Movo Photo Color, White Balance Card Set for Digital Photography (Index Card Sized, 5 X 4)",accessories,Grey Cards,0.8412,0.8714,0.7582,0.0,related_category,image



METHOD: hybrid


Status: ok | {'method': 'hybrid', 'requested_k': 10, 'selected_image_available': True, 'image_fallback_to_text': False, 'candidate_count': 49, 'eligible_distinct_products': 8, 'available_distinct_products': 13211, 'category_backfill_used': True, 'returned': 10, 'compatible_returned': 8, 'related_category_fill_count': 2, 'related_categories_used': ['related_category'], 'result_text_fallback_count': 0}


Rank,Image,ASIN,Title,Group,Category,Score,E5 cosine,CLIP cosine,Category compatibility,Source,Used
1,,B00DMA06AY,"DGK Color Tools DKC-Pro 5"" x 7"" Set of 2 White Balance and Color Calibration Charts with 12% and 18% Gray - Pro Quality - Includes Frame Stand and User Guide",accessories,Color Calibration Charts,0.9268,0.9228,0.6784,1.0,compatible,"text, image"
2,,B07QR6J48J,X-Rite ColorChecker Passport Photo (MSCCPP),accessories,Color Calibration Charts,0.9155,0.8389,0.7630,1.0,compatible,"text, image"
3,,B00EGBIZ2O,"DGK Color Tools Digital Kolor Pro 16:9 Large Color Calibration and Video Chip Chart, 2-Pack",accessories,Color Calibration Charts,0.9117,0.8717,0.6774,1.0,compatible,"text, image"
4,,B08RDK4TPR,Datacolor SpyderCHECKR 24 - Color calibrate your camera for consistent image color across multiple camera systems/lighting conditions. Target color chart has 24 target colors + grey card.,accessories,Color Calibration Charts,0.9014,0.8383,0.6744,1.0,compatible,"text, image"
5,,B0C5N8B857,Calibrite ColorChecker Digital SG (CCDSG),accessories,Color Calibration Charts,0.8982,0.8543,0.6242,1.0,compatible,"text, image"
6,,B07VCTWR5Q,"Pixel Perfect Camera Color Correction Card – (Set of 2) - 4"" x 6"" for Photo and Video - Reference Tool Grey Card Target White Balance Exposure Temperature Color Calibration Chart",accessories,Color Calibration Charts,0.8907,0.8606,0.5647,1.0,compatible,"text, image"
7,,B0C2ZRC4PV,Datacolor Spyder X Capture Pro: Photo Tool Kit for Precision Color Control from Capture Through Editing,accessories,Color Calibration Charts,0.8906,0.8380,0.6060,1.0,compatible,"text, image"
8,,B07QNYHD9H,X-Rite i1Display Pro - Display Calibration (EODIS3),accessories,Color Calibration Charts,0.8897,0.8219,0.6306,1.0,compatible,"text, image"
9,,B073XPTZJW,"Anwenk Grey Card White Balance Card 18% Exposure Photography Card Custom Calibration Camera Checker Video, DSLR and Film",accessories,Grey Cards,0.8822,0.8642,0.8218,0.0,related_category,"text, image"
10,,B00GSC97HG,"Movo Photo Color, White Balance Card Set for Digital Photography (Index Card Sized, 5 X 4)",accessories,Grey Cards,0.8743,0.8714,0.7582,0.0,related_category,"text, image"


## 6 · Catalog-based evaluation

Compute actual diagnostics on the sampled catalog products, without manufacturing relevance labels:

| Area | Measurement |
|---|---|
| Relevance proxies | Mean seed–recommendation E5 cosine and CLIP cosine (CLIP only for valid pairs). These reuse ranking features and are **not independent relevance evidence**. |
| Diversity | Text intra-list diversity = mean `(1 − E5 cosine) / 2` across all result pairs; image diversity uses only valid-image pairs and reports their count. Also report distinct leaf categories and category entropy in nats. Higher diversity can reduce substitutability. |
| Category consistency | Same product-group rate, exact leaf rate, and mean taxonomy compatibility. Strict compatibility applies to primary recommendations; fallback rows can lower consistency. Report fallback counts separately; consistency does not establish relevance or model fit. |
| Coverage | Seeds receiving ≥1 / all sampled seeds; full top-10 lists / all sampled seeds; unique recommended ASINs / catalog size; fill rate = returned / 10. Coverage depends on the sampled seeds, deduplication, and eligible categories. |

Report all-seed and paired image-available subsets, with denominators and per-group breakdowns. Undefined pair/empty-list metrics remain null, not invented zeroes. Coverage and fill rate include empty responses as zero.


In [7]:
def intra_list_diversity(rows, vectors):
    if len(rows) < 2:
        return None, 0
    selected = np.asarray(vectors[rows], dtype='float32')
    cosine = np.clip(selected @ selected.T, -1.0, 1.0)
    upper = np.triu_indices(len(rows), k=1)
    return float(np.mean((1.0 - cosine[upper]) / 2.0)), len(upper[0])

def mean_or_none(values):
    return float(np.mean(values)) if values else None

diagnostics = []
for (seed_asin, method), response in runs.items():
    results = response['results']
    seed = recommender.row_by_id[seed_asin]
    rows = [r['catalog_row'] for r in results]
    image_result_rows = [r for r in rows if image_valid[r]]
    text_ild, text_pairs = intra_list_diversity(rows, e5_vectors)
    image_ild, image_pairs = intra_list_diversity(image_result_rows, image_vectors)
    leaves = Counter(recommender.leaves[r] for r in rows)
    entropy = -sum((n / len(rows)) * math.log(n / len(rows)) for n in leaves.values()) if rows else None
    diagnostics.append({
        'seed_asin': seed_asin, 'seed_catalog_row': int(seed), 'method': method,
        'product_group': text_value(products.iloc[seed].product_group),
        'seed_image_available': bool(image_valid[seed]), 'status': response['status'],
        'returned': len(rows), 'fill_rate@10': len(rows) / TOP_K,
        'category_fallback_count': sum(r['is_category_fallback'] for r in results),
        'category_fallback_rate': mean_or_none([int(r['is_category_fallback']) for r in results]),
        'mean_seed_text_cosine': mean_or_none([r['text_similarity'] for r in results]),
        'mean_seed_image_cosine': mean_or_none([r['image_similarity'] for r in results
                                                if r['image_similarity'] is not None]),
        'seed_image_pairs': sum(r['image_similarity'] is not None for r in results),
        'text_intra_list_diversity': text_ild, 'text_diversity_pairs': text_pairs,
        'image_intra_list_diversity': image_ild, 'image_diversity_pairs': image_pairs,
        'distinct_leaf_categories': len(leaves) if rows else None, 'leaf_entropy_nats': entropy,
        'same_group_rate': mean_or_none([int(recommender.groups[r] == recommender.groups[seed]) for r in rows]),
        'same_leaf_rate': mean_or_none([int(recommender.leaves[r] == recommender.leaves[seed]) for r in rows]),
        'mean_category_compatibility': mean_or_none([r['category_compatibility'] for r in results]),
        'text_only_result_count': sum(r['modalities_used'] == ['text'] for r in results)})
diagnostic_df = pd.DataFrame(diagnostics)

quality_columns = ['mean_seed_text_cosine', 'mean_seed_image_cosine', 'text_intra_list_diversity',
                   'image_intra_list_diversity', 'same_group_rate', 'same_leaf_rate',
                   'mean_category_compatibility', 'distinct_leaf_categories', 'leaf_entropy_nats', 'category_fallback_rate']
summary_rows = []
for subset_name, frame in [
    ('all_seeds', diagnostic_df),
    ('image_available_seeds', diagnostic_df[diagnostic_df.seed_image_available])]:
    for method, group in frame.groupby('method', sort=True):
        seed_ids = set(group.seed_asin)
        exposed = {r['parent_asin'] for asin in seed_ids for r in runs[(asin, method)]['results']}
        summary = {'subset': subset_name, 'method': method, 'sampled_seeds': len(group),
                   'nonempty_seeds': int(group.returned.gt(0).sum()),
                   'full_list_seeds': int(group.returned.eq(TOP_K).sum()),
                   'seed_coverage': float(group.returned.gt(0).mean()),
                   'full_list_rate': float(group.returned.eq(TOP_K).mean()),
                   'mean_fill_rate@10': float(group['fill_rate@10'].mean()),
                   'unique_recommended_products': len(exposed),
                   'catalog_size': len(products), 'sample_catalog_coverage': len(exposed) / len(products)}
        for column in quality_columns:
            summary[column] = float(group[column].mean()) if group[column].notna().any() else None
            summary[column + '_observed_seeds'] = int(group[column].notna().sum())
        summary_rows.append(summary)
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
group_summary_df = diagnostic_df.groupby(['product_group', 'method'], as_index=False).agg(
    seeds=('seed_asin', 'size'), returned_mean=('returned', 'mean'),
    fill_rate=('fill_rate@10', 'mean'), category_fallback_rate=('category_fallback_rate', 'mean'),
    same_group_rate=('same_group_rate', 'mean'),
    same_leaf_rate=('same_leaf_rate', 'mean'),
    text_diversity=('text_intra_list_diversity', 'mean'))
display(group_summary_df)
print('Similarity proxies and category consistency do not replace the human relevance evaluation below.')


,subset,method,sampled_seeds,nonempty_seeds,full_list_seeds,seed_coverage,full_list_rate,mean_fill_rate@10,unique_recommended_products,catalog_size,...,same_leaf_rate,same_leaf_rate_observed_seeds,mean_category_compatibility,mean_category_compatibility_observed_seeds,distinct_leaf_categories,distinct_leaf_categories_observed_seeds,leaf_entropy_nats,leaf_entropy_nats_observed_seeds,category_fallback_rate,category_fallback_rate_observed_seeds
0,all_seeds,hybrid,39,39,39,1.0,1.0,1.0,389,13328,...,0.969231,39,0.981795,39,1.179487,39,0.078188,39,0.015385,39
1,all_seeds,image,39,39,39,1.0,1.0,1.0,390,13328,...,0.920513,39,0.971795,39,1.333333,39,0.144676,39,0.015385,39
2,all_seeds,text,39,39,39,1.0,1.0,1.0,387,13328,...,0.969231,39,0.982051,39,1.205128,39,0.080079,39,0.015385,39
3,image_available_seeds,hybrid,39,39,39,1.0,1.0,1.0,389,13328,...,0.969231,39,0.981795,39,1.179487,39,0.078188,39,0.015385,39
4,image_available_seeds,image,39,39,39,1.0,1.0,1.0,390,13328,...,0.920513,39,0.971795,39,1.333333,39,0.144676,39,0.015385,39
5,image_available_seeds,text,39,39,39,1.0,1.0,1.0,387,13328,...,0.969231,39,0.982051,39,1.205128,39,0.080079,39,0.015385,39


,product_group,method,seeds,returned_mean,fill_rate,category_fallback_rate,same_group_rate,same_leaf_rate,text_diversity
0,accessories,hybrid,3,10.0,1.0,0.2,1.0,0.733333,0.069696
1,accessories,image,3,10.0,1.0,0.2,1.0,0.633333,0.074915
2,accessories,text,3,10.0,1.0,0.2,1.0,0.766667,0.067019
3,camera,hybrid,3,10.0,1.0,0.0,1.0,1.000000,0.041683
4,camera,image,3,10.0,1.0,0.0,1.0,1.000000,0.051048
5,camera,text,3,10.0,1.0,0.0,1.0,1.000000,0.039387
6,desktop,hybrid,3,10.0,1.0,0.0,1.0,0.933333,0.034993
7,desktop,image,3,10.0,1.0,0.0,1.0,0.800000,0.045078
8,desktop,text,3,10.0,1.0,0.0,1.0,1.000000,0.032906
9,headphones,hybrid,3,10.0,1.0,0.0,1.0,1.000000,0.057131


Similarity proxies and category consistency do not replace the human relevance evaluation below.


## 7 · Human relevance evaluation on the same real product pairs

There are no existing item-pair relevance labels in notebooks 01–05. Pool distinct seed/candidate pairs from all three methods, present their real metadata, and leave grades **blank**. The final save cell creates or extends `relevance_judgments.csv` without replacing any existing grades. Label real pairs manually, using metadata/images and product details:

- **0:** unrelated, wrong product role, or unsuitable as a similar alternative.
- **1:** a plausible related alternative with significant differences or uncertain suitability.
- **2:** a strong similar alternative for the selected product's intended role.

Do not label from embedding scores, category agreement, review stars, or popularity alone. Unknown model/connector compatibility is not a proven fit. The pooled CSV omits method names, ranks, and scores to reduce presentation bias.

After labeling, rerun from this section. Compute Precision@10 (grades 1–2 relevant), MRR@10, and pooled nDCG@10 (`2^grade−1`). All-zero judged pools give nDCG 0. Withhold a seed's comparison unless **every returned pair in all three methods** is judged. Report evaluated/unevaluated seed counts, all-seed and image-available comparisons; do not treat unjudged pairs as irrelevant. Empty image responses count as zero ranking metrics once the seed comparison is otherwise fully judged.

These are fixed-setting diagnostics on sampled seeds, not held-out tuning results. No quality advantage is assumed. If weights are later tuned, reserve independent seed families for final evaluation.


In [8]:
JUDGMENTS_PATH = SAVE / 'relevance_judgments.csv'
label_columns = ['catalog_hash', 'seed_asin', 'candidate_asin', 'seed_catalog_row', 'candidate_catalog_row',
                 'seed_title', 'candidate_title', 'seed_category', 'candidate_category', 'grade']
pool = {}
for (seed_asin, method), response in runs.items():
    seed = response['selected_product']
    for item in response['results']:
        pool[(seed_asin, item['parent_asin'])] = {
            'catalog_hash': catalog_hash, 'seed_asin': seed_asin, 'candidate_asin': item['parent_asin'],
            'seed_catalog_row': seed['catalog_row'], 'candidate_catalog_row': item['catalog_row'],
            'seed_title': seed['title'], 'candidate_title': item['title'],
            'seed_category': seed['cat_path'], 'candidate_category': item['cat_path'], 'grade': ''}
pooled = pd.DataFrame(list(pool.values()), columns=label_columns)
if not pooled.empty:
    pooled = pooled.sort_values(['seed_asin', 'candidate_asin']).sample(frac=1, random_state=SEED).reset_index(drop=True)

judgments = pd.DataFrame(columns=label_columns)
if JUDGMENTS_PATH.is_file():
    judgments = pd.read_csv(JUDGMENTS_PATH, dtype=str, keep_default_na=False)
    require(set(label_columns) <= set(judgments), 'Unexpected recommendation judgment schema')
    require(not judgments.duplicated(['seed_asin', 'candidate_asin']).any(), 'Duplicate item-pair judgments')
    require(judgments.catalog_hash.eq(catalog_hash).all(), 'Judgments belong to a different catalog')
    for field in ('seed_asin', 'candidate_asin'):
        require(set(judgments[field]) <= set(asins), f'Unknown {field} in judgments')
    require(judgments.seed_asin.ne(judgments.candidate_asin).all(), 'Self-pairs in judgment file')
    for prefix in ('seed', 'candidate'):
        expected = judgments[prefix + '_asin'].map(recommender.row_by_id)
        actual = pd.to_numeric(judgments[prefix + '_catalog_row'], errors='coerce')
        require(actual.eq(expected).all(), 'Judgment product ID/row mapping mismatch')
    numeric = pd.to_numeric(judgments.grade, errors='coerce')
    require((judgments.grade.eq('') | numeric.isin([0, 1, 2])).all(), 'Grades must be blank, 0, 1, or 2')

old_keys = set(zip(judgments.seed_asin, judgments.candidate_asin))
additional = pooled.loc[[key not in old_keys for key in zip(pooled.seed_asin, pooled.candidate_asin)]].copy()
judgments_to_save = pd.concat([judgments, additional], ignore_index=True)
grades = {
    (r.seed_asin, r.candidate_asin): int(float(r.grade))
    for r in judgments.itertuples(index=False) if r.grade != ''}

coverage_rows, relevance_rows = [], []
for row in sample_rows:
    seed_asin = recommender.ids[row]
    expected = {(seed_asin, item['parent_asin']) for method in METHODS
                for item in runs[(seed_asin, method)]['results']}
    unknown = expected - set(grades)
    coverage_rows.append({'seed_asin': seed_asin, 'pooled_pairs': len(expected),
                          'judged_pairs': len(expected - unknown), 'unknown_pairs': len(unknown),
                          'comparison_complete': len(unknown) == 0 and len(expected) > 0})
    if unknown or not expected:
        continue
    ideal = sorted([g for (seed_id, _), g in grades.items() if seed_id == seed_asin], reverse=True)[:TOP_K]
    idcg = sum((2**g - 1) / math.log2(rank + 2) for rank, g in enumerate(ideal))
    for method in METHODS:
        results = runs[(seed_asin, method)]['results']
        gains = [grades[(seed_asin, r['parent_asin'])] for r in results]
        relevant = [int(g > 0) for g in gains]
        dcg = sum((2**g - 1) / math.log2(rank + 2) for rank, g in enumerate(gains))
        relevance_rows.append({
            'seed_asin': seed_asin, 'method': method, 'product_group': recommender.groups[row],
            'seed_image_available': bool(image_valid[row]), 'precision@10': sum(relevant) / TOP_K,
            'mrr@10': next((1 / rank for rank, value in enumerate(relevant, 1) if value), 0.0),
            'ndcg@10': dcg / idcg if idcg else 0.0})

judgment_coverage_df = pd.DataFrame(coverage_rows)
relevance_df = pd.DataFrame(relevance_rows)
display(judgment_coverage_df)
print('Fully judged seed comparisons:', int(judgment_coverage_df.comparison_complete.sum()), '/', len(sample_rows))
if relevance_df.empty:
    print('Human relevance metrics withheld: label the blank real-product pairs, then rerun this section.')
else:
    print('Judged subset only; inspect coverage before generalizing:')
    display(relevance_df.groupby('method')[['precision@10', 'mrr@10', 'ndcg@10']].mean())
    paired_relevance = relevance_df[relevance_df.seed_image_available]
    if not paired_relevance.empty:
        print('Paired image-available seeds:')
        display(paired_relevance.groupby('method')[['precision@10', 'mrr@10', 'ndcg@10']].mean())
print('New blank pairs to save:', len(additional), '| existing grades preserved.')


,seed_asin,pooled_pairs,judged_pairs,unknown_pairs,comparison_complete
0,B001KNP3MQ,11,0,11,False
1,B07QT88378,15,0,15,False
2,B08FGD53GP,15,0,15,False
3,B000MANEDI,21,0,21,False
4,B088WTLBQR,19,0,19,False
5,B0B1M91G7W,18,0,18,False
6,B089YX5QJ6,22,0,22,False
7,B0BCQD7KWP,18,0,18,False
8,B0CBD98MY3,19,0,19,False
9,B01JP436TS,22,0,22,False


Fully judged seed comparisons: 0 / 39
Human relevance metrics withheld: label the blank real-product pairs, then rerun this section.
New blank pairs to save: 12 | existing grades preserved.


## 8 · Validation checks and reusable API

Run these checks in Colab. They verify input handling, deterministic order, seed/title deduplication, compatible-first ordering, deterministic related-category filling, finite JSON serialization, exactly 10 hybrid results for every valid sampled product, and missing-image behavior. They do not establish relevance.

The `ItemRecommender` constructor takes already-loaded artifacts and ranking settings. `recommend_products(parent_asin, top_k=10)` is a thin hybrid entry point. Initialize the service once in a future FastAPI startup and map explicit invalid/not-found statuses to HTTP responses there. No backend/frontend files are changed.


In [9]:
for bad_id in (None, '', 'bad id', 123):
    require(recommend_products(bad_id)['status'] == 'invalid_product_id', 'Invalid-ID contract failed')
# Generate a syntactically valid ID confirmed absent; do not assume an ASIN is absent.
missing_id = next(f'{i:010d}' for i in range(len(products) + 1) if f'{i:010d}' not in recommender.row_by_id)
require(recommend_products(missing_id)['status'] == 'product_not_found', 'Not-found contract failed')
for invalid_k in (0, -1, 101, True, 3.5):
    try:
        recommend_products(DEMO_ASIN, invalid_k)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid top_k accepted')

for (seed_asin, method), response in runs.items():
    json.dumps(response, allow_nan=False)
    seed = recommender.row_by_id[seed_asin]
    results = response['results']
    rows = [r['catalog_row'] for r in results]
    require(seed not in rows and len(rows) == len(set(rows)), 'Seed/repeated row returned')
    require(len({r['parent_asin'] for r in results}) == len(results), 'Duplicate ASIN returned')
    keys = [recommender.duplicate_keys[r] for r in rows]
    require(recommender.duplicate_keys[seed] not in keys and len(keys) == len(set(keys)),
            'Normalized title duplicates returned')
    require(all(r['parent_asin'] == str(asins[r['catalog_row']]) for r in results), 'ID/row mismatch')
    for item in results:
        tier, relatedness = recommender.category_relation(seed, item['catalog_row'])
        require(item['category_tier'] == tier and item['category_relatedness'] == relatedness,
                'Incorrect category tier/relatedness')
        require(item['is_category_fallback'] == (tier > 0), 'Incorrect fallback flag')
        require((recommender.compatibility(seed, item['catalog_row']) > 0) == (tier == 0),
                'Strict compatibility and fallback classification disagree')
    compatible_count = min(TOP_K, len({recommender.duplicate_keys[r]
                                      for r in recommender.eligible_rows(seed, method)}))
    if response['status'] != 'image_unavailable':
        require(sum(not r['is_category_fallback'] for r in results) == compatible_count,
                'A related-category fallback displaced a compatible product')
    require(all(np.isfinite(r['score']) for r in results), 'Non-finite recommendation score')
    ordering = [(r['category_tier'], -r['score'], r['parent_asin']) for r in results]
    require(ordering == sorted(ordering), 'Tier/score/ASIN order is not deterministic')
    if method == 'image':
        require(all(image_valid[row] for row in rows), 'Image-only returned missing image')
        if not image_valid[seed]:
            require(response['status'] == 'image_unavailable' and not results, 'Image-only silently fell back')
    if method == 'hybrid' and not image_valid[seed]:
        require(all(r['modalities_used'] == ['text'] and r['image_similarity'] is None for r in results),
                'Missing-image hybrid fallback failed')
    if response['status'] != 'image_unavailable':
        available = response['diagnostics']['available_distinct_products']
        require(len(results) == min(TOP_K, available), 'Incorrect global fill count after deduplication')
    require(response == recommender.recommend(seed_asin, TOP_K, method), 'Non-deterministic recommendations')

# Regression: the public hybrid API must fill all 10 slots for EVERY real sampled product,
# including sparse accessory categories and selected products without image embeddings.
for row in sample_rows:
    asin = recommender.ids[row]
    response = recommend_products(asin, top_k=10)
    require(response['status'] == 'ok' and len(response['results']) == 10,
            f'Hybrid returned fewer than 10 recommendations for sampled product {asin}')
    require(response == runs[(asin, 'hybrid')], 'Public API differs from evaluated hybrid output')
    require(response == recommend_products(asin, top_k=10), 'Repeated public API call changed ranking')
print('Exactly 10 hybrid recommendations returned for all', len(sample_rows), 'real catalog samples.')

# Force an empty strict category around a real sampled product. No new vectors/indexes.
# This makes the fallback regression run even when all sampled categories are dense.
probe_seed = sample_rows[0]
probe_catalog = products.copy(deep=False).assign(product_group=products.product_group.copy())
probe_catalog.loc[probe_seed, 'product_group'] = '__isolated_test_group__'
probe_view = ItemRecommender(probe_catalog, e5_vectors, image_vectors, image_valid,
                             e5_index, image_index, image_rows, RANKING)
probe_response = probe_view.recommend(recommender.ids[probe_seed], top_k=10)
require(probe_response['status'] == 'ok' and len(probe_response['results']) == 10,
        'Related-category fallback failed for a seed with no strict alternatives')
require(all(r['is_category_fallback'] for r in probe_response['results']), 'Expected fallback rows only')
require(len({probe_view.duplicate_keys[r['catalog_row']] for r in probe_response['results']}) == 10,
        'Fallback returned duplicate titles')
require(all(r['catalog_row'] != probe_seed for r in probe_response['results']), 'Fallback returned seed')
require(probe_response == probe_view.recommend(recommender.ids[probe_seed], top_k=10),
        'Related-category fallback changed ordering')

# Exercise missing-image handling even if the real catalog has no missing images.
# Temporary availability-mask views share saved arrays/indexes without altering them.
image_seed = next((r for r in sample_rows if image_valid[r]), None)
if image_seed is not None:
    probe = recommender.recommend(recommender.ids[image_seed], method='hybrid')
    if probe['results']:
        candidate = probe['results'][0]['catalog_row']
        view_mask = image_valid.copy()
        view_mask[candidate] = False
        fallback_view = ItemRecommender(products, e5_vectors, image_vectors, view_mask,
                                        e5_index, image_index, image_rows, RANKING)
        scored = fallback_view.score_candidates(image_seed, [candidate], 'hybrid')[0]
        require(scored['modalities_used'] == ['text'] and scored['image_similarity'] is None,
                'Missing candidate image must use text-only evidence')
        view_mask[image_seed] = False
        fallback = fallback_view.recommend(recommender.ids[image_seed], method='hybrid')
        require(all(r['modalities_used'] == ['text'] and r['image_similarity'] is None
                    for r in fallback['results']), 'Missing seed image fallback failed')
        require(fallback_view.recommend(recommender.ids[image_seed], method='image')['status'] == 'image_unavailable',
                'Missing seed image-only handling failed')
print('Colab contract, ranking, duplicate, category, and mapping checks passed.')


Exactly 10 hybrid recommendations returned for all 39 real catalog samples.
Colab contract, ranking, duplicate, category, and mapping checks passed.


## 9 · Save only necessary configuration and evaluation files

Write three stable files directly under `ElectroLens/recommendation_assets/`:

1. `recommendation_config.json`: source references/fingerprints, ranking and compatibility settings, evaluation sampling settings, package versions.
2. `recommendation_evaluation.json`: actual sample IDs, recommendation outputs with catalog rows/ASINs, automatic metrics, judgment coverage, and human metrics when available.
3. `relevance_judgments.csv`: the pooled human-review input/evaluation evidence, preserving existing grades and extra reviewer columns.

Rerunning updates these same files. No embeddings, FAISS indexes, product copies, images, review copies, or precomputed full-catalog recommendation tables are saved. Review images load only when displayed.


In [10]:
def frame_records(frame):
    return json.loads(frame.to_json(orient='records'))

SAVE.mkdir(parents=True, exist_ok=True)
configuration = {
    'schema_version': 2,
    'catalog_hash': catalog_hash, 'n_products': len(products),
    'source_directory': 'search_assets', 'source_files': ASSET_FILES,
    'source_manifest': manifest,
    'source_manifest_sha256': hashlib.sha256((SEARCH / 'manifest.json').read_bytes()).hexdigest(),
    'clip_mapping_sha256': hashlib.sha256(image_rows.astype('<i8').tobytes()).hexdigest(),
    'row_mapping': {'e5': 'FAISS position = products_search catalog_row',
                    'clip': 'clip_image_row_ids.npy[FAISS position] = catalog_row'},
    'product_id_field': 'parent_asin', 'ranking': RANKING,
    'score_formula': '(1-category_weight)*available_modality_weighted_scaled_cosine + category_weight*category_relatedness',
    'ordering': ['category_tier', 'descending score', 'parent_asin'],
    'missing_images': 'hybrid/text use E5; image-only explicitly unavailable',
    'category_policy': {
        'main_devices': 'same product_group required for primary recommendations',
        'main_scores': {'same_leaf': 1.0, 'same_specific_parent': 0.85, 'same_group': 0.60},
        'accessories': 'same full path or same specific parent plus informative leaf overlap',
        'accessory_scores': {'same_path': 1.0, 'compatible_sibling': 0.75},
        'fallback_tiers': ['compatible', 'related_category', 'same_department', 'catalog_similarity'],
        'fallback_relatedness': {'related_category': 0.50, 'same_department': 0.25, 'catalog_similarity': 0.10},
        'fallback_rule': 'fill only after compatible products; exhaust closer taxonomy tiers before broader semantic neighbors',
        'model_fit_verified': False},
    'duplicates': 'exclude seed, repeated ASIN and exact normalized-title duplicates',
    'short_lists': 'only when fewer than top_k distinct alternatives exist in the entire modality-eligible catalog',
    'sampling': {'seed': SEED, 'per_group': SAMPLES_PER_GROUP, 'include_missing_image_examples': True,
                 'top_k': TOP_K, 'methods': list(METHODS), 'fixed_settings_not_tuned': True},
    'packages': {name: version(name) for name in ['numpy', 'pandas', 'pyarrow', 'faiss-cpu']},
    'python': platform.python_version(),
}
report = {
    'catalog_hash': catalog_hash, 'sample_products': frame_records(sample_df),
    'recommendations': [{'seed_asin': asin, 'method': method, **response}
                        for (asin, method), response in runs.items()],
    'per_seed_diagnostics': frame_records(diagnostic_df),
    'summary': frame_records(summary_df), 'per_group_summary': frame_records(group_summary_df),
    'judgment_coverage': frame_records(judgment_coverage_df), 'human_relevance': frame_records(relevance_df),
    'asset_mapping_checks': mapping_checks,
    'limitations': [
        'Sampled group-balanced evaluation is not a traffic-weighted/full-catalog benchmark.',
        'Cosine similarities and category consistency do not independently prove relevance.',
        'Related-category filling can reduce strict category consistency; broad semantic fallback is explicitly flagged.',
        'Unjudged relevance is withheld; complete judged subsets may still have selection bias.',
        'Pooled nDCG ideal rankings do not enumerate all relevant catalog alternatives.',
        'Missing-image weight renormalization affects score distributions.',
        'Taxonomy compatibility does not guarantee connector/model/electrical compatibility.',
        'Normalized-title duplicate checks do not identify every near-duplicate or variant.',
        'No personalization, user interaction evaluation, or independent weight-tuning test is available.'],
}
# Preserve human grades; only append previously unseen blank pairs.
judgments_to_save.to_csv(JUDGMENTS_PATH, index=False)
report['judgments_sha256'] = hashlib.sha256(JUDGMENTS_PATH.read_bytes()).hexdigest()
for filename, content in [('recommendation_config.json', configuration),
                          ('recommendation_evaluation.json', report)]:
    (SAVE / filename).write_text(json.dumps(content, indent=2, allow_nan=False), encoding='utf-8')
print('Saved configuration, evaluation and judgments to:', SAVE)
print('No source search artifacts were written or duplicated.')


Saved configuration, evaluation and judgments to: /content/drive/MyDrive/ElectroLens/recommendation_assets
No source search artifacts were written or duplicated.


## Run in Colab

Run cells in order after mounting the Drive with notebook 03–04 artifacts. CPU is sufficient; model loading is unnecessary. Call `recommend_products(parent_asin, top_k=10)` for the final hybrid response or `recommender.recommend(..., method='text'/'image'/'hybrid')` for comparisons. Inspect displayed products and automatic metrics, save the three files, fill genuine grades in `relevance_judgments.csv`, then rerun sections 7–9 to obtain human relevance metrics.

This notebook has been prepared for static validation; actual recommendation/evaluation results require the Drive assets and a Colab run. No performance or relevance advantage is claimed before those results and human judgments exist. Keep the settings fixed for this evaluation; changing them requires fresh evaluation and, for tuning claims, independent held-out seed families.
